# 📊 Credit Default Prediction

## 🎯 Problema de Negócio
O objetivo é prever a probabilidade de inadimplência de clientes em cobranças mensais.

A inadimplência é definida como pagamentos realizados com atraso igual ou superior a 5 dias,
caracterizando um evento de risco relevante para o negócio.

---

## 🧠 Objetivo Técnico
Desenvolver um modelo de machine learning capaz de estimar a probabilidade de inadimplência
para cada cobrança futura, permitindo priorização de ações.

---

## ⚙️ Estratégia
- Construção da variável target a partir do histórico de pagamentos
- Criação de features comportamentais (lags, rolling e histórico do cliente)
- Uso de validação temporal para simular cenário real de produção
- Modelagem com algoritmo baseado em árvores (LightGBM)
- Avaliação com métricas adequadas para problemas desbalanceados (AUC, KS, AP)

---

## 💼 Aplicação
O modelo permite:
- Priorizar clientes com maior risco
- Otimizar ações de cobrança
- Reduzir custos operacionais
- Apoiar decisões de crédito

## 1. Setup do Ambiente


**Objetivo:** Centralizar todas as bibliotecas utilizadas no projeto, garantindo organização,
           reprodutibilidade e facilidade de manutenção do código.

**Decisão:** As bibliotecas foram escolhidas com base no padrão de mercado para problemas de modelagem preditiva com dados tabulares. Utiliza-se pandas e numpy para manipulação de dados, matplotlib e seaborn para visualização, e LightGBM como algoritmo principal por sua eficiência e desempenho em problemas de classificação.

**Métricas e validação:** Foram importadas métricas como AUC, KS e Precision-Recall para avaliar o modelo sob diferentes perspectivas, especialmente considerando possível desbalanceamento da target. Também foram incluídas ferramentas de calibração para garantir qualidade das probabilidades previstas.


In [1]:

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report,
    log_loss
)

from sklearn.model_selection import train_test_split
from scipy.stats import ks_2samp
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

## 2. Carregamento e Padronização das Bases de Dados


**Objetivo:** Carregar as bases de dados fornecidas e garantir consistência estrutural 
           para viabilizar integrações e transformações ao longo do pipeline.

**Decisão:** As bases foram carregadas separadamente respeitando sua granularidade:

                • cadastral → nível cliente (informações estáticas)
            
                • info → nível mensal (informações dinâmicas)
            
                • pagamentos → histórico com target
            
                • teste → base para inferência

Essa separação permite construir uma base analítica completa via joins controlados.


In [2]:

PATH = '../data/'

cadastral = pd.read_csv(PATH + 'base_cadastral.csv', sep=';')
info = pd.read_csv(PATH + 'base_info.csv', sep=';')
pag = pd.read_csv(PATH + 'base_pagamentos_desenvolvimento.csv', sep=';')
teste = pd.read_csv(PATH + 'base_pagamentos_teste.csv', sep=';')

print("Arquivos carregados com sucesso!")



# Foi realizada a limpeza dos nomes das colunas para evitar inconsistências 
# (em especial espaços em branco), que podem gerar erros silenciosos em merges.


## Limpeza na base
cadastral.columns = cadastral.columns.str.strip()
info.columns = info.columns.str.strip()
pag.columns = pag.columns.str.strip()
teste.columns = teste.columns.str.strip()


# Uma verificação inicial das colunas garante que a estrutura está conforme esperado, evitando problemas nas etapas seguintes do pipeline.

print("\nColunas da base de pagamentos: ")
print(pag.columns)

# Definição centralizada do cutoff temporal
# Ponto único de controle — qualquer alteração aqui se propaga para
# seções 5.2, 6 e 11.1 automaticamente
CUTOFF = '2018-10-01'
print(f"\nCutoff temporal definido: {CUTOFF}")


Arquivos carregados com sucesso!

Colunas da base de pagamentos: 
Index(['ID_CLIENTE', 'SAFRA_REF', 'DATA_EMISSAO_DOCUMENTO', 'DATA_PAGAMENTO',
       'DATA_VENCIMENTO', 'VALOR_A_PAGAR', 'TAXA'],
      dtype='object')

Cutoff temporal definido: 2018-10-01


## 3. Definição da Target de Inadimplência (Regra de Negócio)


**Objetivo:** Construir a variável target de inadimplência a partir da regra de negócio, transformando o problema em uma tarefa de classificação binária.

**Definição:** Considera-se inadimplente todo pagamento realizado com 5 dias ou mais de atraso em relação à data de vencimento.

**Decisão:** A definição da target foi baseada diretamente na regra do negócio, garantindo alinhamento entre o modelo preditivo e o problema real a ser resolvido.

**Tratamentos aplicados:**

    1. Conversão de datas: As colunas de data foram convertidas para datetime para permitir cálculos temporais precisos e evitar inconsistências.

    2. Cálculo de atraso: Foi criada a variável 'dias_atraso', representando o número de dias entre pagamento e vencimento, capturando o comportamento de pagamento do cliente.

    3. Tratamento de ausência de pagamento: Valores nulos em DATA_PAGAMENTO foram interpretados como não pagamento e tratados como atraso elevado (999 dias), garantindo correta classificação como inadimplente.

    4. Criação da target: 
                    A variável target foi definida como:
                            • 1 → atraso >= 5 dias
                            • 0 → caso contrário

    5. Variável auxiliar: Foi criada a variável 'atraso_grave' (>= 30 dias) para suportar análises exploratórias e possíveis estratégias futuras de segmentação de risco.

**Validação:** Foi realizada uma inspeção inicial da variável target para verificar sua distribuição, permitindo avaliar o nível de desbalanceamento do problema.


In [3]:

# Conversão de datas
pag['DATA_PAGAMENTO'] = pd.to_datetime(pag['DATA_PAGAMENTO'], errors='coerce')
pag['DATA_VENCIMENTO'] = pd.to_datetime(pag['DATA_VENCIMENTO'], errors='coerce')

# Cálculo de atraso
pag['dias_atraso'] = (pag['DATA_PAGAMENTO'] - pag['DATA_VENCIMENTO']).dt.days

# Tratamento de não pagamento
# Se não pagou = vira atraso alto
pag['dias_atraso'] = pag['dias_atraso'].fillna(999)

# Criação da Target
pag['target'] = (pag['dias_atraso'] >= 5).astype(int)
pag['atraso_grave'] = (pag['dias_atraso'] >= 30).astype(int)

# Validação
print("\nAmostra dos dados:")
pag[['DATA_VENCIMENTO', 'DATA_PAGAMENTO', 'dias_atraso', 'target']].head(10)


print("\nDistribuição da target:")
print(pag['target'].value_counts(normalize=True))


Amostra dos dados:

Distribuição da target:
target
0    0.92978
1    0.07022
Name: proportion, dtype: float64


## Construção da variável target

A variável alvo foi definida como inadimplência quando o pagamento ocorre com atraso igual ou superior a 5 dias.

Pagamentos ausentes foram tratados como inadimplência, assumindo atraso elevado (999 dias), o que representa clientes que não realizaram o pagamento.

## 4. Construção da Base Final
Objetivo: Construir a base analítica final consolidando todas as fontes de dados necessárias para modelagem.

Estratégia: A base de pagamentos foi utilizada como tabela principal (fato), pois contém a variável target e representa o evento de interesse. A partir dela, foram incorporadas informações complementares:

        • info → dados dinâmicos por cliente e período (join por ID_CLIENTE + SAFRA_REF)
        • cadastral → dados estáticos do cliente (join por ID_CLIENTE)
Decisão: Foi utilizado left join para preservar todos os registros da base de pagamentos, garantindo que nenhuma observação com target seja perdida.

Ordenação: A base foi ordenada por cliente e tempo, o que é essencial para etapas posteriores de engenharia de features baseadas em histórico (lags, rolling, etc.).

In [4]:
df = pag.merge(info, on=['ID_CLIENTE', 'SAFRA_REF'], how='left')
df = df.merge(cadastral, on='ID_CLIENTE', how='left')

#  ordenação correta
df = df.sort_values(['ID_CLIENTE', 'SAFRA_REF', 'DATA_VENCIMENTO'])

df.head()

,ID_CLIENTE,SAFRA_REF,DATA_EMISSAO_DOCUMENTO,DATA_PAGAMENTO,DATA_VENCIMENTO,VALOR_A_PAGAR,TAXA,dias_atraso,target,atraso_grave,RENDA_MES_ANTERIOR,NO_FUNCIONARIOS,DATA_CADASTRO,DDD,FLAG_PF,SEGMENTO_INDUSTRIAL,DOMINIO_EMAIL,PORTE,CEP_2_DIG
166,8784237149961904,2018-08,2018-08-17,2018-09-04,2018-09-04,100616.1,5.99,0,0,0,NaN,NaN,2011-02-14,11,NaN,Comércio,HOTMAIL,PEQUENO,27
168,8784237149961904,2018-08,2018-08-23,2018-09-10,2018-09-10,94062.8,5.99,0,0,0,NaN,NaN,2011-02-14,11,NaN,Comércio,HOTMAIL,PEQUENO,27
169,8784237149961904,2018-08,2018-08-23,2018-09-08,2018-09-10,102686.1,5.99,-2,0,0,NaN,NaN,2011-02-14,11,NaN,Comércio,HOTMAIL,PEQUENO,27
167,8784237149961904,2018-08,2018-08-22,2018-09-11,2018-09-11,89552.8,5.99,0,0,0,NaN,NaN,2011-02-14,11,NaN,Comércio,HOTMAIL,PEQUENO,27
170,8784237149961904,2018-08,2018-08-24,2018-09-11,2018-09-11,51393.0,5.99,0,0,0,NaN,NaN,2011-02-14,11,NaN,Comércio,HOTMAIL,PEQUENO,27


## 4.1 Validações e Sanity Checks
Objetivo: Validar a integridade da base após os merges, garantindo que a estrutura esteja consistente antes da etapa de engenharia de features.

Decisão: Foram realizadas verificações essenciais para evitar erros silenciosos que podem comprometer a modelagem, especialmente em problemas com múltiplas fontes de dados.

Validações realizadas:

• Estrutura da base: Verificação de tipo e dimensão para garantir que os dados foram carregados e combinados corretamente.

• Consistência de colunas: Confirmação da presença das variáveis esperadas após os merges.

• Duplicidade: Checagem de duplicidade na chave (ID_CLIENTE + SAFRA_REF), assegurando granularidade correta (uma linha por cliente por período).

• Valores ausentes: Análise do percentual de missing para identificar possíveis necessidades de tratamento na etapa de feature engineering.

• Consistência temporal: Inspeção de registros de um cliente ao longo do tempo para garantir ordenação e coerência do histórico.
Importância: Essas validações são fundamentais para garantir a qualidade dos dados e evitar propagação de erros nas etapas seguintes, especialmente em features baseadas em histórico.

In [5]:
print(type(df))
print(df.shape)

print("\nColunas:")
print(df.columns)

print("\nDuplicidades (ID_CLIENTE + SAFRA_REF):")
df[['ID_CLIENTE', 'SAFRA_REF']].duplicated().sum()

print("\nTop colunas com missing:")
df.isnull().mean().sort_values(ascending=False).head(10)

print("\nExemplo de registros:")
df[['ID_CLIENTE','SAFRA_REF']].head(10)


print("\nExemplo de histórico de um cliente:")
display(
    df[df['ID_CLIENTE'] == df['ID_CLIENTE'].iloc[0]][
        ['SAFRA_REF','DATA_VENCIMENTO']
    ].head(10)
)

<class 'pandas.core.frame.DataFrame'>
(77414, 19)

Colunas:
Index(['ID_CLIENTE', 'SAFRA_REF', 'DATA_EMISSAO_DOCUMENTO', 'DATA_PAGAMENTO',
       'DATA_VENCIMENTO', 'VALOR_A_PAGAR', 'TAXA', 'dias_atraso', 'target',
       'atraso_grave', 'RENDA_MES_ANTERIOR', 'NO_FUNCIONARIOS',
       'DATA_CADASTRO', 'DDD', 'FLAG_PF', 'SEGMENTO_INDUSTRIAL',
       'DOMINIO_EMAIL', 'PORTE', 'CEP_2_DIG'],
      dtype='object')

Duplicidades (ID_CLIENTE + SAFRA_REF):

Top colunas com missing:

Exemplo de registros:

Exemplo de histórico de um cliente:


,SAFRA_REF,DATA_VENCIMENTO
166,2018-08,2018-09-04
168,2018-08,2018-09-10
169,2018-08,2018-09-10
167,2018-08,2018-09-11
170,2018-08,2018-09-11
171,2018-09,2018-09-24
172,2018-09,2018-09-24
173,2018-09,2018-09-25
174,2018-09,2018-09-27
2794,2018-09,2018-10-03
